# dbt setup, run, test

Run from Databricks Repos after **Pull**. Requires bronze tables and gold dimensions from `07_dimensional/`.

Set `dbt/profiles.yml` in the `dbt/` folder (gitignored — not pushed from PC).

In [ ]:
%pip install -q "dbt-databricks>=1.9.0,<2.0.0"

In [ ]:
import json
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path("/Workspace/Repos")  # adjust if your repo path differs
# Common pattern: find repo by walking from cwd
cwd = Path.cwd()
DBT_DIR = cwd / "dbt" if (cwd / "dbt" / "dbt_project.yml").exists() else cwd.parent / "dbt"
if not (DBT_DIR / "dbt_project.yml").exists():
    for p in [cwd, *cwd.parents]:
        if (p / "dbt" / "dbt_project.yml").exists():
            DBT_DIR = p / "dbt"
            break

PROFILES = DBT_DIR / "profiles.yml"
if not PROFILES.exists():
    example = (DBT_DIR / "profiles.yml.example").read_text()
    PROFILES.write_text(example)

# Profile target name (dev/prod) != UC schema prefix (dbt_dev/dbt_prod) — see profiles.yml
TARGET_SCHEMA = {"dev": "dbt_dev", "prod": "dbt_prod"}


def run_dbt(*args, target="dev"):
    cmd = ["dbt", *args, "--project-dir", str(DBT_DIR), "--profiles-dir", str(DBT_DIR), "--target", target]
    print(" ".join(cmd))
    out = subprocess.run(cmd, capture_output=True, text=True, check=False)
    print(out.stdout)
    if out.returncode != 0:
        print(out.stderr)
        raise RuntimeError(f"dbt failed: {' '.join(args)}")
    return out.stdout


def resolve_dbt_table(
    table_name: str,
    *,
    target: str = "dev",
    catalog: str = "globalmart",
    custom_layer: str | None = "marts",
) -> str:
    """Resolve UC three-part name for a dbt object (e.g. dbt_dev_marts.fact_sales_incremental)."""
    schema = TARGET_SCHEMA.get(target, target)
    candidates = []
    if custom_layer:
        candidates.append(f"{catalog}.{schema}_{custom_layer}.{table_name}")
    candidates.extend([
        f"{catalog}.{schema}.{table_name}",
        f"{catalog}.snapshots.{table_name}",
    ])
    for fq in candidates:
        try:
            spark.table(fq)
            print(f"Resolved {table_name} -> {fq}")
            return fq
        except Exception:
            continue
    raise RuntimeError(f"Could not find {table_name}. Tried: {candidates}")


print("dbt project:", DBT_DIR)

In [ ]:
run_dbt("debug", target="dev")
run_dbt("debug", target="prod")

In [ ]:
run_dbt("source", "freshness", target="dev")

In [ ]:
run_dbt("run", target="dev")
run_dbt("test", target="dev")

In [ ]:
run_dbt("run", "--select", "fact_sales_incremental", target="dev")
fact_table = resolve_dbt_table("fact_sales_incremental", target="dev", custom_layer="marts")
initial = spark.table(fact_table).count()
run_dbt("run", "--select", "fact_sales_incremental", target="dev")
rerun = spark.table(fact_table).count()

report = {
    "task": "dbt_incremental_fact",
    "table": fact_table,
    "row_count_after_initial": int(initial),
    "row_count_after_rerun": int(rerun),
    "idempotent_rerun": initial == rerun,
}
print(json.dumps(report, indent=2))

In [ ]:
run_dbt("snapshot", target="dev")
snap_table = resolve_dbt_table("snap_customers", target="dev", custom_layer=None)
snap_count = spark.table(snap_table).count()
print(json.dumps({"task": "dbt_snapshot", "table": snap_table, "snap_customers_rows": int(snap_count)}, indent=2))